# RAG

In [12]:
import requests
import xml.etree.ElementTree as ET
import json
import os
import time

API = "http://export.arxiv.org/api/query"
THEME_KEYWORDS = ["exoplanet", "planet detection", "habitability", "TRAPPIST", "TESS"]
# THEME_KEYWORDS = ["exoplanet", "TRAPPIST"]
OUTPUT_FILE = "exoplanet_data.json"
MAX_DATA_SIZE = 2000

def get_exoplanet_data_xml(start=0, batch_size=100):
    search_terms = " OR ".join([f'all:{keyword}' for keyword in THEME_KEYWORDS])
    query = f"search_query=({search_terms})&start={start}&max_results={batch_size}"
    url = f"{API}?{query}"
    
    response = requests.get(url)
    response.raise_for_status()
    
    root = ET.fromstring(response.content)
    return root

def read_xml(xml_root: ET.Element, output):
    namespace = {"atom": "http://www.w3.org/2005/Atom"}
    rows_added = 0
    
    for entry in xml_root.findall('atom:entry', namespace):
        authors = entry.findall('atom:author', namespace)
        
        categories = entry.findall('atom:category', namespace)
        categories_list = [cat.get('term') for cat in categories] if categories else []
        
        title_elem = entry.find('atom:title', namespace)
        id_elem = entry.find('atom:id', namespace)
        published_elem = entry.find('atom:published', namespace)
        summary_elem = entry.find('atom:summary', namespace)
        
        if title_elem is None or id_elem is None or published_elem is None or summary_elem is None:
            continue
        
        author_names = []
        for author in authors:
            name_elem = author.find('atom:name', namespace)
            if name_elem is not None and name_elem.text:
                author_names.append(name_elem.text)
        
        row = {
            "title": title_elem.text.strip() if title_elem.text else "",
            "id": id_elem.text if id_elem.text else "",
            "published": published_elem.text if published_elem.text else "",
            "summary": summary_elem.text.strip() if summary_elem.text else "",
            "authors": author_names,
            "categories": categories_list
        }
        output.append(row)
        rows_added += 1
        
    return rows_added

def fetch_exoplanet_data(output_file=OUTPUT_FILE):
    batch_size = 100
    total_rows = 0
    data = []
    
    print("Start fetching exoplanet data...")
    
    while total_rows < MAX_DATA_SIZE:
        print(f"Fetching data from {total_rows} to {total_rows + batch_size}")
        
        try:
            xml_root = get_exoplanet_data_xml(total_rows, batch_size)
            rows_added = read_xml(xml_root, data)
            
            if rows_added == 0:
                print("No more data found, exiting...")
                break
                
            total_rows += rows_added
            print(f"Added {rows_added} rows to {output_file}, total rows: {total_rows}")
            time.sleep(1)
            
        except Exception as e:
            print(f"Error fetching data: {e}")
            break
    
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(data, f, indent=4, ensure_ascii=False)
    
    print(f"Data saved to {output_file}")
    
    return data

articles = fetch_exoplanet_data()

Start fetching exoplanet data...
Fetching data from 0 to 100
Added 100 rows to exoplanet_data.json, total rows: 100
Fetching data from 100 to 200
Added 100 rows to exoplanet_data.json, total rows: 200
Fetching data from 200 to 300
Added 100 rows to exoplanet_data.json, total rows: 300
Fetching data from 300 to 400
Added 100 rows to exoplanet_data.json, total rows: 400
Fetching data from 400 to 500
Added 100 rows to exoplanet_data.json, total rows: 500
Fetching data from 500 to 600
Added 100 rows to exoplanet_data.json, total rows: 600
Fetching data from 600 to 700
Added 100 rows to exoplanet_data.json, total rows: 700
Fetching data from 700 to 800
No more data found, exiting...
Data saved to exoplanet_data.json


In [13]:
print(f"Всего загружено публикаций: {len(articles)}")
print(f"Первые 3 публикации:")

for i, article in enumerate(articles[:3]):
    print(f"\n{i+1}. {article['title']}")
    print(f"   Авторы: {', '.join(article['authors'][:3])}{'...' if len(article['authors']) > 3 else ''}")
    print(f"   Категории: {', '.join(article['categories'])}")
    print(f"   Опубликовано: {article['published'][:10]}")
    print(f"   Аннотация: {article['summary'][:200]}...")


Всего загружено публикаций: 700
Первые 3 публикации:

1. TESS Habitable Zone Star Catalog
   Авторы: L. Kaltenegger, J. Pepper, K. Stassun...
   Категории: astro-ph.EP
   Опубликовано: 2019-03-27
   Аннотация: We present the Transiting Exoplanet Survey Satellite (TESS) Habitable Zone
Stars Catalog, a list of 1822 nearby stars with a TESS magnitude brighter than
T = 12 and reliable distances from Gaia DR2, a...

2. Around which stars can TESS detect Earth-like planets? The Revised TESS
  Habitable Zone Catalog
   Авторы: L. Kaltenegger, J. Pepper, P. M. Christodoulou...
   Категории: astro-ph.EP, astro-ph.IM, astro-ph.SR
   Опубликовано: 2021-01-19
   Аннотация: In the search for life in the cosmos, NASA's Transiting Exoplanet Survey
Satellite (TESS) mission has already monitored about 74% of the sky for
transiting extrasolar planets, including potentially ha...

3. Analyzing the Habitable Zones of Circumbinary Planets Using Machine
  Learning
   Авторы: Zhihui Kong, Jonathan H. Jiang, 

In [ ]:
from collections import Counter

all_categories = []
for article in articles:
    all_categories.extend(article['categories'])

# - atstro-ph.EP - Earth and planetary physics
# - astro-ph.SR - Sollar ans Stellar astrophysics
# - astro-ph.IM - Instrumentation and Methods for Astrophysics
# - astro-ph.GA - Astrophysics of Galaxies

category_counts = Counter(all_categories)
print("Топ-10 категорий публикаций:")
for category, count in category_counts.most_common(10):
    print(f"  {category}: {count} публикаций")


Топ-10 категорий публикаций:
  astro-ph.EP: 668 публикаций
  astro-ph.SR: 173 публикаций
  astro-ph.IM: 159 публикаций
  astro-ph.GA: 20 публикаций
  astro-ph: 16 публикаций
  physics.ao-ph: 14 публикаций
  cs.LG: 11 публикаций
  physics.space-ph: 5 публикаций
  physics.geo-ph: 4 публикаций
  physics.pop-ph: 3 публикаций


## Разбиение текста на чанки

Разобьем статьт на более мелкие части - чанки


In [27]:
import re

CHUNK_SIZE = 500
OVERLAP_SIZE = 50

def clean_text(text):
    # лишние пробелы и переносы строк
    text = re.sub(r'\s+', ' ', text.strip())
    # специальные символы, оставляя только буквы, цифры и пунктуацию
    text = re.sub(r'[^\w\s\.\,\;\:\!\?\-\(\)]', ' ', text)
    return text

def split_into_chunks(text: str, chunk_size: int = CHUNK_SIZE, overlap_size: int = OVERLAP_SIZE):
    words = text.split()
    
    if len(words) <= chunk_size:
        return [text]
    
    chunks = []
    start = 0
    
    while start < len(words):
        end = min(start + chunk_size, len(words))

        chunk_words = words[start:end]
        chunk_text = ' '.join(chunk_words)
        chunks.append(chunk_text)
        
        if end >= len(words):
            break
            
        start = end - overlap_size
        
        if start >= end:
            break
    
    return chunks

def create_chunks_from_articles(articles):
    all_chunks = []
    
    for article_idx, article in enumerate(articles):
        title = article.get('title', '')
        summary = article.get('summary', '')

        full_text = f"{title}. {summary}"
        full_text = clean_text(full_text)

        chunks = split_into_chunks(full_text)
        
        for chunk_idx, chunk_text in enumerate(chunks):
            chunk_data = {
                'chunk_id': f"{article_idx}_{chunk_idx}",
                'article_id': article_idx,
                'chunk_index': chunk_idx,
                'text': chunk_text,
                'article_title': title,
                'article_authors': article.get('authors', []),
                'article_categories': article.get('categories', []),
                'article_published': article.get('published', ''),
                'word_count': len(chunk_text.split())
            }
            all_chunks.append(chunk_data)
    
    return all_chunks

chunks = create_chunks_from_articles(articles)

print(f"Создано чанков: {len(chunks)}")
print(f"В среднем чанков на статью: {len(chunks) / len(articles):.2f}")


Создано чанков: 700
В среднем чанков на статью: 1.00


In [29]:
from collections import Counter

print(f"Общее количество чанков: {len(chunks)}")

chunk_sizes = [chunk['word_count'] for chunk in chunks]
print(f"Размер чанков:")
print(f"  - Минимальный: {min(chunk_sizes)} слов")
print(f"  - Максимальный: {max(chunk_sizes)} слов") 
print(f"  - Средний: {sum(chunk_sizes)/len(chunk_sizes):.1f} слов")

articles_chunk_count = Counter([chunk['article_id'] for chunk in chunks])
chunk_counts = list(articles_chunk_count.values())

print(f"\nЧанков на статью:")
print(f"  - Минимум: {min(chunk_counts)} чанков")
print(f"  - Максимум: {max(chunk_counts)} чанков")
print(f"  - Среднее: {sum(chunk_counts)/len(chunk_counts):.1f} чанков")


Общее количество чанков: 700
Размер чанков:
  - Минимальный: 56 слов
  - Максимальный: 339 слов
  - Средний: 232.1 слов

Чанков на статью:
  - Минимум: 1 чанков
  - Максимум: 1 чанков
  - Среднее: 1.0 чанков


In [ ]:
import json

CHUNKS_FILE = "exoplanet_chunks.json"
with open(CHUNKS_FILE, 'w', encoding='utf-8') as f:
    json.dump(chunks, f, indent=2, ensure_ascii=False)

print(f"Чанки сохранены в файл: {CHUNKS_FILE}")
# Создаем также простой список текстов для TF-IDF анализа
chunk_texts = [chunk['text'] for chunk in chunks]
print(f"Подготовлен список из {len(chunk_texts)} текстов чанков для анализа")


Чанки сохранены в файл: exoplanet_chunks.json
Размер файла: 1.33 MB
Подготовлен список из 700 текстов чанков для анализа



**Размер чанков:** Все чанки получились меньше 500 слов (максимум 339, средний 232 слова). 

Можно было бы уменьшить размер чанка, но кажется, что средний размер в 232 слова и так является оптимальным



## Векторизация текста и построение FAISS индекса

Для RAG-поиска преобразуем тексты в векторные представления (эмбеддинги) и создадим индекс для быстрого поиска.


In [31]:
from sentence_transformers import SentenceTransformer

print("Загрузка модели эмбеддингов...")
model = SentenceTransformer('all-MiniLM-L6-v2')
print(f"Модель загружена. Размерность эмбеддингов: {model.get_sentence_embedding_dimension()}")


Загрузка модели эмбеддингов...
Модель загружена. Размерность эмбеддингов: 384


In [36]:
def create_embeddings(chunk_texts):
    batch_size = 10
    embeddings = []

    for i in range(0, len(chunk_texts), batch_size):
        batch = chunk_texts[i:i+batch_size]
        batch_embeddings = model.encode(batch, show_progress_bar=False, convert_to_numpy=True)
        embeddings.append(batch_embeddings)
        
        if (i // batch_size + 1) % 10 == 0:
            print(f"Обработано {min(i + batch_size, len(chunk_texts))} из {len(chunk_texts)} чанков")

    # Объединение в одну матрицу
    return np.vstack(embeddings)

In [37]:
chunk_texts = [chunk['text'] for chunk in chunks]
embeddings = create_embeddings(chunk_texts)
print(f"Форма матрицы эмбеддингов: {embeddings.shape}")
print(f"Размерность каждого эмбеддинга: {embeddings.shape[1]}")


Обработано 100 из 700 чанков
Обработано 200 из 700 чанков
Обработано 300 из 700 чанков
Обработано 400 из 700 чанков
Обработано 500 из 700 чанков
Обработано 600 из 700 чанков
Обработано 700 из 700 чанков
Форма матрицы эмбеддингов: (700, 384)
Размерность каждого эмбеддинга: 384


Поистройка индекса

In [ ]:
import faiss

faiss.normalize_L2(embeddings)
dimension = embeddings.shape[1]
index = faiss.IndexFlatIP(dimension)

index.add(embeddings.astype('float32'))

print(f"Индекс построен!")
print(f"Количество векторов в индексе: {index.ntotal}")
print(f"Размерность векторов: {index.d}")


Индекс построен!
Количество векторов в индексе: 700
Размерность векторов: 384


In [43]:
import numpy as np
import pickle

FIASS_INDEX_PATH = "faiss_index.bin"
METADATA_PATH = "chunks_metadata.pkl"

def save_faiss_index(index, index_path=FIASS_INDEX_PATH, metadata_path=METADATA_PATH):
    print("Сохранение FAISS индекса и метаданных...")

    faiss.write_index(index, index_path)
    print(f"FAISS индекс сохранен в: {index_path}")

    with open(metadata_path, 'wb') as f:
        pickle.dump(chunks, f)
    print(f"Метаданные чанков сохранены в: {metadata_path}")

def load_faiss_index(index_path=FIASS_INDEX_PATH, metadata_path=METADATA_PATH):
    loaded_index = faiss.read_index(index_path)
    with open(metadata_path, 'rb') as f:
        loaded_chunks = pickle.load(f)
    
    return loaded_index, loaded_chunks


In [44]:
save_faiss_index(index)

Сохранение FAISS индекса и метаданных...
FAISS индекс сохранен в: faiss_index.bin
Метаданные чанков сохранены в: chunks_metadata.pkl


Теперь напишем саму функцию для семантического поиска

In [45]:
def search_similar_chunks(query_text, top_k=5):
    query_embedding = model.encode([query_text], convert_to_numpy=True)
    faiss.normalize_L2(query_embedding)
    
    distances, indices = index.search(query_embedding.astype('float32'), top_k)
    
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        chunk = chunks[idx]
        results.append({
            'chunk_id': chunk['chunk_id'],
            'similarity': float(dist),
            'text': chunk['text'],
            'article_title': chunk['article_title'],
            'article_authors': chunk['article_authors'],
            'article_categories': chunk['article_categories'],
            'word_count': chunk['word_count']
        })
    
    return results


In [51]:

test_queries = [
    "What are habitable exoplanets?",
    "How does TESS detect planets?",
    "What is the TRAPPIST system?",
]

for query in test_queries:
    print(f"Запрос: '{query}'")
    results = search_similar_chunks(query, top_k=1)
    
    for i, result in enumerate(results, 1):
        print(f"\n{i}. Сходство: {result['similarity']:.4f}")
        print(f"   Статья: {result['article_title']}")
        print(f"   Авторы: {', '.join(result['article_authors'][:2])}")
        print(f"   Текст: {result['text'][:200]}...")

    print("\n" + "=" * 80 + "\n")

Запрос: 'What are habitable exoplanets?'

1. Сходство: 0.7225
   Статья: Characterizing Exoplanet Habitability
   Авторы: Ravi kumar Kopparapu, Eric T. Wolf
   Текст: Characterizing Exoplanet Habitability. Habitability is a measure of an environment s potential to support life, and a habitable exoplanet supports liquid water on its surface. However, a planet s succ...


Запрос: 'How does TESS detect planets?'

1. Сходство: 0.7130
   Статья: The Transiting Exoplanet Survey Satellite
   Авторы: Joshua N. Winn
   Текст: The Transiting Exoplanet Survey Satellite. A transiting planet invites us to measure its size, mass, orbital parameters, atmospheric composition, and other characteristics. But the invitation can only...


Запрос: 'What is the TRAPPIST system?'

1. Сходство: 0.3703
   Статья: Planet-Planet Tides in the TRAPPIST-1 System
   Авторы: Jason T. Wright
   Текст: Planet-Planet Tides in the TRAPPIST-1 System. The star TRAPPIST-1 hosts a system of seven transiting, terrestrial exop